# 02 · Train Road Damage Detector (yolo_rdd.pt)

> **OWNER:** Member A (M1 · road defects)
> **PREREQUISITES:** `01_prepare_rdd2022.ipynb` complete — `data/rdd2022_india/data.yaml`
> must exist and you must have looked at its contact sheet.
> **EXPECTED RUNTIME:** ~2 hours on a T4 (both runs combined; each run is roughly 1 hour).
> **OUTPUTS:** `models/yolo_rdd.pt`, `models/yolo_rdd.onnx`, `models/MODEL_CARD_rdd.md`,
> a chosen confidence threshold written into `services/edge/defects/config.py`,
> and failure-analysis contact sheets.

**Next notebook:** `07_evaluate_all.ipynb` (after Member B finishes 03-06 too).

In [ ]:
# Cell 1/3 — minimal bootstrap (Colab vs local). No repo imports yet: on a
# fresh Colab runtime nothing has been cloned, so this cell is deliberately
# self-contained and only prepares sys.path so `common/` becomes importable.
import subprocess
import sys
from pathlib import Path


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


IN_COLAB = _in_colab()
REPO_URL = "https://github.com/ad8thya/SmartIndiaHackathon.git"

if IN_COLAB:
    REPO_ROOT = Path("/content/SmartIndiaHackathon")
    if not REPO_ROOT.exists():
        print(f"cloning {REPO_URL} -> {REPO_ROOT}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f"{REPO_ROOT} already present locally on this runtime")
else:
    _here = Path.cwd().resolve()
    _candidates = [c for c in (_here, *_here.parents) if (c / "pyproject.toml").exists() and (c / "notebooks").exists()]
    if not _candidates:
        raise RuntimeError(
            "Could not find the repo root (looked for pyproject.toml + notebooks/ "
            f"walking up from {_here}). Run this notebook from inside the repo checkout."
        )
    REPO_ROOT = _candidates[0]

for _p in (str(REPO_ROOT), str(REPO_ROOT / "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"Colab: {IN_COLAB}")
print(f"repo root: {REPO_ROOT}")

In [ ]:
# Cell 2/3 — install the ML extras. Quiet; ~60-90s on a fresh Colab runtime,
# near-instant if already installed (pip no-ops on a satisfied requirement).
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[ml]"],
    check=True,
)
print("ml extras installed")

In [ ]:
# Cell 3/3 — full environment setup (Drive mount + DATA_ROOT/MODEL_ROOT) and
# an import check: torch version + CUDA availability, so a broken environment
# fails here, not forty minutes into a training run.
from common import colab as colab_mod

env = colab_mod.setup_environment()
REPO_ROOT, DATA_ROOT, MODEL_ROOT = env["repo_root"], env["data_root"], env["model_root"]

import ultralytics

print(f"ultralytics {ultralytics.__version__}")
gpu_info = colab_mod.gpu_report()

## Step 1 — Assert data.yaml matches the frozen indices

In [ ]:
import yaml

from common import constants

DATA_YAML_PATH = DATA_ROOT / "rdd2022_india" / "data.yaml"
if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_YAML_PATH} not found — run 01_prepare_rdd2022.ipynb first. "
        "On Colab, DATA_ROOT is Drive-backed (MyDrive/urban-twin-ml/data/), so if you already "
        "ran 01 in a different session, check you mounted the same Google account's Drive."
    )
with open(DATA_YAML_PATH) as f:
    data_yaml = yaml.safe_load(f)

constants.assert_class_order(data_yaml["names"], constants.RDD_DETECTION_NAMES, "rdd")

## Step 2 — Baseline run

`yolo11s.pt`, imgsz 640, epochs 50, batch auto, AdamW, lr0 1e-3, patience 15, seed 42.

Augmentation choices, deliberately:
- `mosaic=1.0`, `mixup=0.1` — standard YOLO regularization for a dataset this size.
- `hsv_v=0.4` — Indian road lighting varies hugely (harsh midday sun to monsoon
  overcast); a wide value-channel jitter is doing real work here, not padding.
- `fliplr=0.5` — a crack looks the same left-right, this is free data.
- `flipud=0.0` — roads have a fixed orientation relative to the camera (sky up,
  road down). A vertical flip teaches the model nothing real about a pothole
  and just wastes training signal on an augmentation that never occurs at inference.

**Resume-on-reconnect:** `project=RUNS_DIR` points at the Drive-backed `MODEL_ROOT`, and ultralytics writes `last.pt`/`best.pt` after every epoch — so a dropped Colab runtime loses at most the epoch in progress, not the run. The training cells below check for an existing `last.pt` under `RUNS_DIR/<name>/weights/` and resume from it automatically instead of restarting from `yolo11s.pt`.

In [ ]:
import random

import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RUNS_DIR = MODEL_ROOT / "runs" / "rdd"

TRAIN_KWARGS = dict(
    imgsz=640,
    epochs=50,
    batch=-1,  # ultralytics auto-batch; see 00's recommended_batch_size if you want to pin one
    optimizer="AdamW",
    lr0=1e-3,
    patience=15,
    seed=SEED,
    mosaic=1.0,
    mixup=0.1,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
    project=str(RUNS_DIR),
    exist_ok=True,
)


def _train_or_resume(name, data_yaml_path):
    """Resume from a Drive-backed checkpoint if one exists — a dropped Colab
    runtime should cost at most the in-progress epoch, not the whole run."""
    last_ckpt = RUNS_DIR / name / "weights" / "last.pt"
    if last_ckpt.exists():
        print(f"found existing checkpoint at {last_ckpt} — resuming from it")
        model = YOLO(str(last_ckpt))
        results = model.train(resume=True)
    else:
        print(f"no existing checkpoint for '{name}' — starting fresh from yolo11s.pt")
        model = YOLO("yolo11s.pt")
        results = model.train(data=str(data_yaml_path), name=name, **TRAIN_KWARGS)
    return model, results


baseline_model, baseline_results = _train_or_resume("baseline", DATA_YAML_PATH)
print(f"baseline run saved to {baseline_results.save_dir}")

## Step 3 — Second run addressing D40 (POTHOLE) imbalance

Same hyperparameters, plus class weighting via ultralytics' `cls` loss gain
is global (not per-class) — the practical lever ultralytics exposes is
**oversampling pothole-containing images** in the training set. Both runs are
kept and reported; do not silently pick one and discard the other, the
before/after comparison is itself evidence for the deck.

In [ ]:
import shutil

D40_INDEX = 3  # constants.RDD_CLASSES[3] == "D40"
# Local/ephemeral, not Drive: this is thousands of small duplicated files, which is
# slow to write over Drive's FUSE mount and trivially regenerable (a few minutes from
# the Drive-backed original) if the session drops — it doesn't need to survive a restart.
OVERSAMPLED_DIR = Path("/content/rdd2022_india_oversampled") if env["colab"] else (DATA_ROOT / "rdd2022_india_oversampled")
OVERSAMPLED_IMAGES = OVERSAMPLED_DIR / "images" / "train"
OVERSAMPLED_LABELS = OVERSAMPLED_DIR / "labels" / "train"
OVERSAMPLED_IMAGES.mkdir(parents=True, exist_ok=True)
OVERSAMPLED_LABELS.mkdir(parents=True, exist_ok=True)

TRAIN_IMAGES_DIR = DATA_ROOT / "rdd2022_india" / "images" / "train"
TRAIN_LABELS_DIR = DATA_ROOT / "rdd2022_india" / "labels" / "train"
OVERSAMPLE_FACTOR = 3  # duplicate every image containing a pothole this many extra times

n_originals, n_duplicates = 0, 0
for label_path in TRAIN_LABELS_DIR.glob("*.txt"):
    classes_present = {int(line.split()[0]) for line in label_path.read_text().splitlines() if line.strip()}
    image_path = next((p for p in TRAIN_IMAGES_DIR.glob(f"{label_path.stem}.*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")), None)
    if image_path is None:
        continue
    shutil.copy2(image_path, OVERSAMPLED_IMAGES / image_path.name)
    shutil.copy2(label_path, OVERSAMPLED_LABELS / label_path.name)
    n_originals += 1
    if D40_INDEX in classes_present:
        for k in range(1, OVERSAMPLE_FACTOR):
            shutil.copy2(image_path, OVERSAMPLED_IMAGES / f"{image_path.stem}_dup{k}{image_path.suffix}")
            shutil.copy2(label_path, OVERSAMPLED_LABELS / f"{label_path.stem}_dup{k}.txt")
            n_duplicates += 1

print(f"train set: {n_originals} original images, {n_duplicates} pothole duplicates added ({OVERSAMPLE_FACTOR}x oversample)")

# val/test stay untouched — copy them over unchanged so this is a valid data.yaml on its own
for split in ("val", "test"):
    shutil.copytree(DATA_ROOT / "rdd2022_india" / "images" / split, OVERSAMPLED_DIR / "images" / split, dirs_exist_ok=True)
    shutil.copytree(DATA_ROOT / "rdd2022_india" / "labels" / split, OVERSAMPLED_DIR / "labels" / split, dirs_exist_ok=True)

oversampled_yaml = {**data_yaml, "path": str(OVERSAMPLED_DIR)}
oversampled_yaml_path = OVERSAMPLED_DIR / "data.yaml"
oversampled_yaml_path.write_text(yaml.safe_dump(oversampled_yaml, sort_keys=False))
constants.assert_class_order(oversampled_yaml["names"], constants.RDD_DETECTION_NAMES, "rdd-oversampled")

In [ ]:
oversampled_model, oversampled_results = _train_or_resume("oversampled", oversampled_yaml_path)
print(f"oversampled run saved to {oversampled_results.save_dir}")

## Step 4 — Evaluate BOTH runs on TEST (not val)

The POTHOLE row is called out separately — that's the number that ends up in the deck.

In [ ]:
from common import evaluate

print("=" * 70)
print("BASELINE — test split")
print("=" * 70)
baseline_table = evaluate.per_class_table(baseline_model, str(DATA_YAML_PATH), split="test")
print(baseline_table.to_string(index=False))
print()
print(">>> POTHOLE row:")
print(baseline_table[baseline_table["class"] == "POTHOLE"].to_string(index=False))

print()
print("=" * 70)
print("OVERSAMPLED — test split (evaluated against the ORIGINAL, non-duplicated test set)")
print("=" * 70)
oversampled_table = evaluate.per_class_table(oversampled_model, str(DATA_YAML_PATH), split="test")
print(oversampled_table.to_string(index=False))
print()
print(">>> POTHOLE row:")
print(oversampled_table[oversampled_table["class"] == "POTHOLE"].to_string(index=False))

## Step 5 — Confidence sweep, choose favouring PRECISION

Sweep 0.1-0.7 on whichever run scored better on POTHOLE recall in Step 4 — set
`BEST_MODEL` below once you've looked at the table. Then pick a threshold that
**favours precision over recall**:

A missed pothole is caught on the next bus pass — the fleet re-drives every
corridor daily, so a false negative here is a delayed detection, not a lost
one. A false positive that survives fusion and reaches `CONFIRMED` dispatches
a repair crew to nothing, which is a real cost with no next-pass correction.
Multi-bus consensus (`services/cloud/consensus`) is a second filter
downstream of this threshold, but it corroborates repeated sightings — it
does not fix a systematically over-confident single-frame detector, so this
threshold should still lean conservative on its own.

In [ ]:
BEST_MODEL = oversampled_model  # <-- set to baseline_model or oversampled_model based on Step 4's POTHOLE row

sweep_table = evaluate.confidence_sweep(BEST_MODEL, str(DATA_YAML_PATH), split="test", thresholds=(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7))
print(sweep_table.to_string(index=False))

CHOSEN_CONFIDENCE = 0.5  # <-- set after reading the table above; favour the row where precision is high without recall collapsing
CONFIDENCE_RATIONALE = (
    "A missed pothole is caught on the bus's next pass over the same route; a false "
    "positive that reaches CONFIRMED dispatches a repair crew to nothing. Multi-bus "
    "consensus corroborates repeat sightings downstream but does not correct a "
    "systematically over-confident detector, so this threshold leans precision-first "
    f"on its own. Chosen from the sweep table: conf={CHOSEN_CONFIDENCE}."
)
print()
print(CONFIDENCE_RATIONALE)

## Step 5b — write the chosen threshold into services/edge/defects/config.py

That file is M1-owned; this notebook is M1's, so this is a same-owner edit.

In [ ]:
import re

config_path = REPO_ROOT / "services" / "edge" / "defects" / "config.py"
config_text = config_path.read_text()
new_config_text, n_subs = re.subn(
    r"DEFECT_CONF_THRESHOLD: float = [0-9.]+",
    f"DEFECT_CONF_THRESHOLD: float = {CHOSEN_CONFIDENCE}",
    config_text,
)
if n_subs != 1:
    raise RuntimeError(f"expected exactly 1 match for DEFECT_CONF_THRESHOLD in {config_path}, got {n_subs} — check the file wasn't already edited")

config_path.write_text(new_config_text)
print(f"DEFECT_CONF_THRESHOLD -> {CHOSEN_CONFIDENCE} written to {config_path}")

## Step 6 — Export + latency

In [ ]:
from common import export

BEST_PT_PATH = MODEL_ROOT / constants.MODEL_FILES["rdd"]
shutil.copy2(Path(BEST_MODEL.trainer.best), BEST_PT_PATH)  # trainer.best = the run's best.pt checkpoint
onnx_path = export.export_onnx(BEST_PT_PATH, imgsz=640, opset=12)

latency = export.benchmark_latency(BEST_PT_PATH, onnx_path, imgsz=640)
export.print_latency_table(latency)

## Step 7 — Failure analysis

Contact sheets of the 20 worst false positives and 20 worst false negatives.
M1 must be able to say out loud what the model gets wrong — this is that evidence.

In [ ]:
TEST_IMAGES_DIR = DATA_ROOT / "rdd2022_india" / "images" / "test"
TEST_LABELS_DIR = DATA_ROOT / "rdd2022_india" / "labels" / "test"

worst_fp = evaluate.worst_predictions(BEST_MODEL, TEST_IMAGES_DIR, TEST_LABELS_DIR, constants.RDD_DETECTION_NAMES, n=20, mode="false_positive", conf=CHOSEN_CONFIDENCE)
worst_fn = evaluate.worst_predictions(BEST_MODEL, TEST_IMAGES_DIR, TEST_LABELS_DIR, constants.RDD_DETECTION_NAMES, n=20, mode="false_negative", conf=CHOSEN_CONFIDENCE)

print(f"worst false positives: {len(worst_fp)} found")
for img_path, note, score in worst_fp[:5]:
    print(f"  {img_path.name}: {note}")
print(f"worst false negatives: {len(worst_fn)} found")
for img_path, note, score in worst_fn[:5]:
    print(f"  {img_path.name}: {note}")

In [ ]:
import shutil as _shutil

FAILURE_DIR = MODEL_ROOT / "failure_analysis_rdd"
FAILURE_DIR.mkdir(parents=True, exist_ok=True)

for label, findings in (("false_positives", worst_fp), ("false_negatives", worst_fn)):
    subdir = FAILURE_DIR / label
    subdir.mkdir(exist_ok=True)
    for img_path, note, _score in findings:
        _shutil.copy2(img_path, subdir / img_path.name)
    (subdir / "notes.txt").write_text("\n".join(f"{p.name}: {n}" for p, n, _ in findings))

sheet_fp = contact_sheet.render_contact_sheet(
    images_dir=FAILURE_DIR / "false_positives",
    labels_dir=TEST_LABELS_DIR,
    class_names=constants.RDD_DETECTION_NAMES,
    output_path=FAILURE_DIR / "worst_false_positives.png",
    n=min(20, len(worst_fp)) or 1,
) if worst_fp else None

sheet_fn = contact_sheet.render_contact_sheet(
    images_dir=FAILURE_DIR / "false_negatives",
    labels_dir=TEST_LABELS_DIR,
    class_names=constants.RDD_DETECTION_NAMES,
    output_path=FAILURE_DIR / "worst_false_negatives.png",
    n=min(20, len(worst_fn)) or 1,
) if worst_fn else None
print("failure-analysis contact sheets written under", FAILURE_DIR)

## Step 8 — Model card

In [ ]:
from common import model_card

metrics_md = evaluate.per_class_table(BEST_MODEL, str(DATA_YAML_PATH), split="test", conf=CHOSEN_CONFIDENCE).to_markdown(index=False)

model_card.render_model_card(
    model_name="yolo_rdd.pt — Road Damage Detector",
    owner="M1",
    base_weights="yolo11s.pt",
    dataset="RDD2022 India subset",
    dataset_size={"train": len(data_splits["train"]), "val": len(data_splits["val"]), "test": len(data_splits["test"])},
    class_names=constants.RDD_DETECTION_NAMES,
    hyperparameters={
        "imgsz": 640, "epochs": 50, "optimizer": "AdamW", "lr0": 1e-3, "patience": 15,
        "seed": SEED, "mosaic": 1.0, "mixup": 0.1, "hsv_v": 0.4, "fliplr": 0.5, "flipud": 0.0,
        "run_chosen": "oversampled" if BEST_MODEL is oversampled_model else "baseline",
    },
    metrics_table_md=metrics_md,
    confidence_chosen=CHOSEN_CONFIDENCE,
    confidence_rationale=CONFIDENCE_RATIONALE,
    latency=latency,
    caveats=[
        "Trained on the India subset of RDD2022 only — generalization to road "
        "surfaces/lighting outside the dataset's collection regions is untested.",
        "D40 (POTHOLE) remains the rarest class even after oversampling — see the "
        "baseline-vs-oversampled comparison in this notebook for the actual delta.",
    ],
    output_path=MODEL_ROOT / "MODEL_CARD_rdd.md",
)

---
### What this notebook produced
- `models/yolo_rdd.pt`, `models/yolo_rdd.onnx`
- `models/MODEL_CARD_rdd.md`
- `services/edge/defects/config.py`'s `DEFECT_CONF_THRESHOLD` updated to the chosen value
- `models/failure_analysis_rdd/` — worst FP/FN contact sheets

### Next
`07_evaluate_all.ipynb` — after Member B (M4) finishes 03-06.